# Phase 4: Portfolio Construction & Risk Management  
## Notebook 04_04 — Mean-Variance Optimization with Ledoit-Wolf Shrinkage

### Research question

How do we build a mean-variance portfolio optimizer that is less fragile by stabilising covariance estimates with Ledoit-Wolf-style shrinkage and validating allocations out of sample?

This notebook follows:

```text
04_01_multi_asset_return_engine
04_02_capm_and_performance_attribution
04_03_volatility_targeting_and_position_sizing
```

The previous notebook sized positions using volatility forecasts. This notebook introduces full portfolio optimisation:

$$
\max_w \quad w^\top \mu - \frac{\lambda}{2}w^\top\Sigma w
$$

subject to portfolio constraints.

It covers:

1. mean-variance optimisation intuition;
2. sample covariance instability;
3. covariance condition numbers and eigenvalues;
4. Ledoit-Wolf shrinkage;
5. covariance shrinkage fallback if `sklearn` is unavailable;
6. global minimum variance portfolio;
7. maximum Sharpe portfolio;
8. long-only constrained optimisation;
9. efficient frontier;
10. shrinkage versus sample covariance comparison;
11. sensitivity to expected returns;
12. turnover and transaction costs;
13. walk-forward out-of-sample test;
14. realised risk attribution;
15. constraint diagnostics;
16. limitations and research extensions.

The key idea:

> Mean-variance optimisation is powerful but extremely input-sensitive. Shrinking covariance estimates often matters more than drawing a beautiful in-sample frontier.

## 1. Mean-variance optimisation

For asset weights:

$$
w =
\begin{bmatrix}
w_1\\
w_2\\
\vdots\\
w_N
\end{bmatrix}
$$

expected returns:

$$
\mu
$$

and covariance matrix:

$$
\Sigma
$$

portfolio expected return:

$$
\mu_p = w^\top \mu
$$

portfolio variance:

$$
\sigma_p^2 = w^\top\Sigma w
$$

A basic mean-variance utility is:

$$
U(w)=w^\top\mu-\frac{\lambda}{2}w^\top\Sigma w
$$

where $\lambda$ is risk aversion.

This model is elegant, but fragile because $\mu$ and $\Sigma$ are estimated with error.

## 2. Why covariance shrinkage?

Sample covariance:

$$
\hat\Sigma_{sample}
$$

can be noisy, especially when:

- the number of assets is large;
- the lookback window is short;
- assets are highly correlated;
- regimes change;
- returns are heavy-tailed.

Noisy covariance matrices create unstable optimised weights.

A shrinkage estimator blends sample covariance with a structured target:

$$
\begin{aligned}
\hat\Sigma_{shrink} &= \delta F \\
&\quad + (1-\delta)\hat\Sigma_{sample}
\end{aligned}
$$

where:

- $F$ is a target matrix;
- $\delta\in[0,1]$ is the shrinkage intensity.

Ledoit-Wolf estimates $\delta$ from data.

This notebook uses `sklearn.covariance.LedoitWolf` when available, with a transparent fallback otherwise.

## 3. Optimisation problem types

### Global minimum variance

$$
\min_w \quad w^\top \Sigma w
$$

subject to:

$$
\sum_i w_i=1
$$

and optionally:

$$
w_i\ge0
$$

### Maximum Sharpe ratio

$$
\max_w \quad \frac{w^\top\mu-r_f} {\sqrt{w^\top\Sigma w}}
$$

### Target-return efficient frontier

$$
\min_w \quad w^\top\Sigma w
$$

subject to:

$$
w^\top\mu = \mu^*
$$

$$
\sum_i w_i=1
$$

This notebook focuses on long-only constrained portfolios because unconstrained portfolios can become extreme.

## 4. Imports and configuration

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from sklearn.covariance import LedoitWolf
    SKLEARN_AVAILABLE = True
except Exception:
    SKLEARN_AVAILABLE = False

try:
    from scipy.optimize import minimize
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False

SKLEARN_AVAILABLE, SCIPY_AVAILABLE

In [ ]:
@dataclass(frozen=True)
class MVOConfig:
    n_days: int = 2_200
    n_assets: int = 12
    train_fraction: float = 0.65
    annualisation: int = 252
    estimation_window: int = 252
    rebalance_step: int = 21
    risk_free_rate_ann: float = 0.02
    max_weight: float = 0.25
    min_weight: float = 0.00
    risk_aversion: float = 6.0
    transaction_cost_bps: float = 3.0
    seed: int = 42


config = MVOConfig()
config

## 5. Synthetic multi-asset return data

We simulate a realistic-ish asset universe with:

- multiple asset classes;
- correlated factors;
- true expected returns;
- true covariance;
- regime shocks;
- noisy realised returns.

Because this is synthetic, we can compare estimated covariance against the true covariance.

In [ ]:
def simulate_asset_returns(config: MVOConfig) -> tuple[pd.DataFrame, pd.DataFrame, np.ndarray, np.ndarray]:
    rng = np.random.default_rng(config.seed)

    dates = pd.bdate_range("2015-01-01", periods=config.n_days)

    assets = [
        "US_EQ", "EU_EQ", "JP_EQ", "EM_EQ",
        "US_BOND", "EU_BOND",
        "GOLD", "OIL",
        "FX_CARRY", "TREND",
        "REIT", "BTC_PROXY"
    ]

    asset_class = {
        "US_EQ": "equity",
        "EU_EQ": "equity",
        "JP_EQ": "equity",
        "EM_EQ": "equity",
        "US_BOND": "bond",
        "EU_BOND": "bond",
        "GOLD": "commodity",
        "OIL": "commodity",
        "FX_CARRY": "fx",
        "TREND": "alternative",
        "REIT": "real_asset",
        "BTC_PROXY": "crypto"
    }

    annual_means = np.array([0.070, 0.060, 0.045, 0.080, 0.025, 0.020, 0.035, 0.045, 0.030, 0.055, 0.060, 0.120])
    annual_vols = np.array([0.160, 0.180, 0.170, 0.230, 0.060, 0.070, 0.180, 0.320, 0.100, 0.120, 0.190, 0.650])

    corr = np.array([
        [1.00, 0.78, 0.65, 0.70,-0.25,-0.20, 0.05, 0.25, 0.10, 0.05, 0.60, 0.35],
        [0.78, 1.00, 0.62, 0.68,-0.22,-0.25, 0.05, 0.20, 0.15, 0.05, 0.55, 0.30],
        [0.65, 0.62, 1.00, 0.60,-0.18,-0.15, 0.02, 0.18, 0.12, 0.04, 0.45, 0.25],
        [0.70, 0.68, 0.60, 1.00,-0.20,-0.18, 0.08, 0.30, 0.20, 0.05, 0.50, 0.38],
        [-0.25,-0.22,-0.18,-0.20, 1.00, 0.72, 0.15,-0.10,-0.05, 0.15,-0.10,-0.20],
        [-0.20,-0.25,-0.15,-0.18, 0.72, 1.00, 0.10,-0.10,-0.05, 0.12,-0.08,-0.15],
        [0.05, 0.05, 0.02, 0.08, 0.15, 0.10, 1.00, 0.22, 0.06, 0.18, 0.15, 0.16],
        [0.25, 0.20, 0.18, 0.30,-0.10,-0.10, 0.22, 1.00, 0.12, 0.10, 0.22, 0.28],
        [0.10, 0.15, 0.12, 0.20,-0.05,-0.05, 0.06, 0.12, 1.00, 0.10, 0.08, 0.10],
        [0.05, 0.05, 0.04, 0.05, 0.15, 0.12, 0.18, 0.10, 0.10, 1.00, 0.05, 0.08],
        [0.60, 0.55, 0.45, 0.50,-0.10,-0.08, 0.15, 0.22, 0.08, 0.05, 1.00, 0.30],
        [0.35, 0.30, 0.25, 0.38,-0.20,-0.15, 0.16, 0.28, 0.10, 0.08, 0.30, 1.00],
    ])

    daily_means = annual_means / config.annualisation
    daily_vols = annual_vols / np.sqrt(config.annualisation)
    true_cov_daily = np.outer(daily_vols, daily_vols) * corr

    returns = np.zeros((config.n_days, len(assets)))
    regime = np.zeros(config.n_days, dtype=int)

    transition = np.array([[0.985, 0.015], [0.055, 0.945]])

    for t in range(config.n_days):
        if t > 0:
            regime[t] = rng.choice([0, 1], p=transition[regime[t - 1]])

        vol_multiplier = 1.0 if regime[t] == 0 else 2.2
        cov_t = true_cov_daily * vol_multiplier**2

        shock = rng.multivariate_normal(daily_means, cov_t)

        # Stress event.
        if rng.uniform() < 0.006:
            shock[0:4] -= rng.uniform(0.025, 0.080, size=4)
            shock[4:6] += rng.uniform(0.002, 0.020, size=2)
            shock[11] -= rng.uniform(0.050, 0.180)

        returns[t] = shock

    returns_df = pd.DataFrame(returns, columns=assets)
    returns_df.insert(0, "date", dates)
    returns_df["regime"] = regime

    metadata = pd.DataFrame({
        "asset": assets,
        "asset_class": [asset_class[a] for a in assets],
        "true_annual_mean": annual_means,
        "true_annual_vol": annual_vols,
    })

    return returns_df, metadata, daily_means, true_cov_daily


returns_df, asset_metadata, true_mu_daily, true_cov_daily = simulate_asset_returns(config)
asset_cols = asset_metadata["asset"].tolist()
returns = returns_df.set_index("date")[asset_cols]

returns_df.head(), asset_metadata

In [ ]:
plt.figure(figsize=(12, 6))
for asset in asset_cols:
    plt.plot(returns.index, (1 + returns[asset]).cumprod(), label=asset, alpha=0.75)
plt.title("Synthetic Asset Growth of 1")
plt.xlabel("Date")
plt.ylabel("Growth")
plt.legend(ncol=3)
plt.show()

## 6. Train/test split

We estimate optimisation inputs on the train period and test portfolios out of sample.

This is crucial because in-sample efficient frontiers often exaggerate performance.

In [ ]:
split_idx = int(len(returns) * config.train_fraction)

train_returns = returns.iloc[:split_idx].copy()
test_returns = returns.iloc[split_idx:].copy()

split_metadata = {
    "n_total_days": int(len(returns)),
    "n_train_days": int(len(train_returns)),
    "n_test_days": int(len(test_returns)),
    "train_start": str(train_returns.index[0]),
    "train_end": str(train_returns.index[-1]),
    "test_start": str(test_returns.index[0]),
    "test_end": str(test_returns.index[-1]),
}

pd.Series(split_metadata)

## 7. Estimate expected returns

Expected returns are notoriously noisy.

This notebook compares covariance estimators, but still shows that mean estimates drive unstable optimisation.

We compute:

$$
\hat\mu = \text{sample mean daily return}
$$

annualised as:

$$
252\hat\mu
$$

Later, we shrink expected returns toward a conservative grand mean.

In [ ]:
sample_mu_daily = train_returns.mean().to_numpy()
sample_mu_ann = sample_mu_daily * config.annualisation

mu_estimates = pd.DataFrame({
    "asset": asset_cols,
    "sample_mu_ann": sample_mu_ann,
    "true_mu_ann": true_mu_daily * config.annualisation,
})

mu_estimates["estimation_error_ann"] = mu_estimates["sample_mu_ann"] - mu_estimates["true_mu_ann"]

mu_estimates.sort_values("sample_mu_ann", ascending=False)

In [ ]:
plt.figure(figsize=(10, 6))
x = np.arange(len(asset_cols))
plt.bar(x - 0.2, mu_estimates["sample_mu_ann"], width=0.4, label="sample")
plt.bar(x + 0.2, mu_estimates["true_mu_ann"], width=0.4, label="true synthetic")
plt.axhline(0, linestyle="--")
plt.xticks(x, asset_cols, rotation=45, ha="right")
plt.title("Expected Return Estimates")
plt.ylabel("Annualised return")
plt.legend()
plt.tight_layout()
plt.show()

## 8. Sample covariance and diagnostics

Sample covariance:

$$
\hat\Sigma_{sample} = \frac{1}{T-1} \sum_t (r_t-\bar r)(r_t-\bar r)^\top
$$

Diagnostics:

- eigenvalues;
- condition number;
- correlation matrix;
- comparison with true covariance.

In [ ]:
sample_cov_daily = train_returns.cov().to_numpy()
true_cov_ann = true_cov_daily * config.annualisation
sample_cov_ann = sample_cov_daily * config.annualisation

def covariance_diagnostics(cov, label):
    eigvals = np.linalg.eigvalsh(cov)
    condition_number = float(eigvals.max() / max(eigvals.min(), 1e-12))

    return {
        "estimator": label,
        "min_eigenvalue": float(eigvals.min()),
        "max_eigenvalue": float(eigvals.max()),
        "condition_number": condition_number,
        "trace": float(np.trace(cov)),
        "avg_variance": float(np.trace(cov) / cov.shape[0])
    }


sample_cov_diag = covariance_diagnostics(sample_cov_ann, "sample_covariance")
sample_cov_diag

In [ ]:
sample_corr = train_returns.corr()

plt.figure(figsize=(9, 8))
plt.imshow(sample_corr.values, vmin=-1, vmax=1)
plt.colorbar(label="Correlation")
plt.xticks(range(len(asset_cols)), asset_cols, rotation=90)
plt.yticks(range(len(asset_cols)), asset_cols)
plt.title("Sample Correlation Matrix")
plt.tight_layout()
plt.show()

eigs = np.linalg.eigvalsh(sample_cov_ann)

plt.figure(figsize=(10, 5))
plt.plot(np.arange(1, len(eigs) + 1), eigs, marker="o")
plt.title("Sample Covariance Eigenvalues")
plt.xlabel("Eigenvalue index")
plt.ylabel("Annualised eigenvalue")
plt.show()

## 9. Ledoit-Wolf shrinkage covariance

When available, use:

```python
sklearn.covariance.LedoitWolf
```

Fallback:

$$
\begin{aligned}
\hat\Sigma_{fallback} &= \delta F \\
&\quad + (1-\delta)\hat\Sigma_{sample}
\end{aligned}
$$

with:

$$
F=\bar\sigma^2 I
$$

This fallback is not the full Ledoit-Wolf estimator, but it preserves the shrinkage idea and keeps the notebook runnable.

In [ ]:
def estimate_ledoit_wolf_covariance(train_returns: pd.DataFrame):
    X = train_returns.to_numpy()

    if SKLEARN_AVAILABLE:
        lw = LedoitWolf().fit(X)
        cov = lw.covariance_
        shrinkage = float(lw.shrinkage_)
        method = "sklearn_ledoit_wolf"
    else:
        sample = np.cov(X, rowvar=False, ddof=1)
        avg_var = np.trace(sample) / sample.shape[0]
        target = avg_var * np.eye(sample.shape[0])
        shrinkage = 0.25
        cov = shrinkage * target + (1 - shrinkage) * sample
        method = "fallback_fixed_identity_shrinkage"

    return cov, shrinkage, method


lw_cov_daily, lw_shrinkage, lw_method = estimate_ledoit_wolf_covariance(train_returns)
lw_cov_ann = lw_cov_daily * config.annualisation

lw_diag = covariance_diagnostics(lw_cov_ann, "ledoit_wolf_shrinkage")
lw_diag["shrinkage_intensity"] = lw_shrinkage
lw_diag["method"] = lw_method

pd.Series(lw_diag)

In [ ]:
cov_diag_table = pd.DataFrame([
    sample_cov_diag,
    lw_diag,
    covariance_diagnostics(true_cov_ann, "true_synthetic_covariance")
])

cov_diag_table

In [ ]:
sample_eigs = np.linalg.eigvalsh(sample_cov_ann)
lw_eigs = np.linalg.eigvalsh(lw_cov_ann)
true_eigs = np.linalg.eigvalsh(true_cov_ann)

plt.figure(figsize=(10, 6))
plt.plot(sample_eigs, marker="o", label="sample")
plt.plot(lw_eigs, marker="x", label="Ledoit-Wolf")
plt.plot(true_eigs, marker="s", label="true synthetic")
plt.title("Covariance Eigenvalues")
plt.xlabel("Eigenvalue index")
plt.ylabel("Annualised eigenvalue")
plt.legend()
plt.show()

## 10. Expected return shrinkage

Mean-variance optimisation is often more sensitive to $\mu$ than $\Sigma$.

We create a conservative expected-return vector:

$$
\begin{aligned}
\hat\mu_{shrunk} &= \rho \bar\mu \\
&\quad + (1-\rho)\hat\mu
\end{aligned}
$$

where $\bar\mu$ is the cross-sectional average.

This reduces extreme bets driven by noisy sample means.

In [ ]:
def shrink_expected_returns(mu_daily, shrinkage=0.50):
    grand_mean = np.mean(mu_daily)
    return shrinkage * grand_mean + (1 - shrinkage) * mu_daily


shrunk_mu_daily = shrink_expected_returns(sample_mu_daily, shrinkage=0.50)

mu_shrink_table = pd.DataFrame({
    "asset": asset_cols,
    "sample_mu_ann": sample_mu_daily * config.annualisation,
    "shrunk_mu_ann": shrunk_mu_daily * config.annualisation,
    "true_mu_ann": true_mu_daily * config.annualisation,
})

mu_shrink_table.sort_values("shrunk_mu_ann", ascending=False)

## 11. Optimisation utilities

We implement:

1. portfolio return;
2. portfolio volatility;
3. Sharpe ratio;
4. constraint-aware optimisation with SciPy when available;
5. fallback random-search optimiser when SciPy is unavailable.

Constraints:

$$
\sum_i w_i=1
$$

$$
w_{min}\le w_i\le w_{max}
$$

In [ ]:
def portfolio_return(w, mu):
    return float(np.asarray(w) @ np.asarray(mu))


def portfolio_variance(w, cov):
    w = np.asarray(w)
    return float(w.T @ cov @ w)


def portfolio_volatility(w, cov):
    return float(np.sqrt(max(portfolio_variance(w, cov), 0.0)))


def normalise_long_only(w, min_weight=0.0, max_weight=1.0):
    w = np.asarray(w, dtype=float)
    w = np.clip(w, min_weight, max_weight)

    if w.sum() <= 0:
        w = np.ones_like(w) / len(w)
    else:
        w = w / w.sum()

    # Reclip once more if needed.
    w = np.clip(w, min_weight, max_weight)
    w = w / w.sum()

    return w


def optimise_long_only(mu, cov, objective="min_var", target_return=None, risk_aversion=5.0, min_weight=0.0, max_weight=0.25, seed=42):
    n = len(mu)
    bounds = [(min_weight, max_weight)] * n
    x0 = np.ones(n) / n

    constraints = [{"type": "eq", "fun": lambda w: np.sum(w) - 1.0}]

    if target_return is not None:
        constraints.append({"type": "eq", "fun": lambda w: w @ mu - target_return})

    if objective == "min_var":
        obj = lambda w: w.T @ cov @ w
    elif objective == "max_sharpe":
        obj = lambda w: -(w @ mu) / max(np.sqrt(w.T @ cov @ w), 1e-12)
    elif objective == "mean_variance":
        obj = lambda w: -(w @ mu - 0.5 * risk_aversion * (w.T @ cov @ w))
    else:
        raise ValueError("Unknown objective.")

    if SCIPY_AVAILABLE:
        res = minimize(obj, x0=x0, method="SLSQP", bounds=bounds, constraints=constraints, options={"maxiter": 500, "ftol": 1e-10})
        if res.success:
            return normalise_long_only(res.x, min_weight, max_weight), {"success": True, "message": res.message, "method": "scipy_slsqp"}

    # Fallback random search.
    rng = np.random.default_rng(seed)
    best_w = x0
    best_val = obj(best_w)

    for _ in range(20_000):
        raw = rng.random(n)
        w = raw / raw.sum()
        w = normalise_long_only(w, min_weight, max_weight)

        if target_return is not None and abs(w @ mu - target_return) > 0.002 / config.annualisation:
            continue

        val = obj(w)
        if val < best_val:
            best_val = val
            best_w = w

    return best_w, {"success": False, "message": "fallback_random_search_or_failed_scipy", "method": "fallback_random_search"}


def weight_table(weights_dict):
    return pd.DataFrame(weights_dict, index=asset_cols)

## 12. Optimised portfolios

We compute:

1. equal-weight portfolio;
2. sample-covariance GMV;
3. Ledoit-Wolf GMV;
4. sample-covariance max Sharpe;
5. Ledoit-Wolf max Sharpe;
6. Ledoit-Wolf mean-variance utility portfolio;
7. shrunk-mean Ledoit-Wolf max Sharpe.

This isolates the role of covariance shrinkage and mean shrinkage.

In [ ]:
n = len(asset_cols)
equal_w = np.ones(n) / n

sample_gmv_w, sample_gmv_info = optimise_long_only(
    sample_mu_daily,
    sample_cov_daily,
    objective="min_var",
    min_weight=config.min_weight,
    max_weight=config.max_weight,
    seed=config.seed
)

lw_gmv_w, lw_gmv_info = optimise_long_only(
    sample_mu_daily,
    lw_cov_daily,
    objective="min_var",
    min_weight=config.min_weight,
    max_weight=config.max_weight,
    seed=config.seed
)

sample_sharpe_w, sample_sharpe_info = optimise_long_only(
    sample_mu_daily,
    sample_cov_daily,
    objective="max_sharpe",
    min_weight=config.min_weight,
    max_weight=config.max_weight,
    seed=config.seed
)

lw_sharpe_w, lw_sharpe_info = optimise_long_only(
    sample_mu_daily,
    lw_cov_daily,
    objective="max_sharpe",
    min_weight=config.min_weight,
    max_weight=config.max_weight,
    seed=config.seed
)

lw_mv_w, lw_mv_info = optimise_long_only(
    sample_mu_daily,
    lw_cov_daily,
    objective="mean_variance",
    risk_aversion=config.risk_aversion,
    min_weight=config.min_weight,
    max_weight=config.max_weight,
    seed=config.seed
)

lw_shrunk_mu_sharpe_w, lw_shrunk_mu_sharpe_info = optimise_long_only(
    shrunk_mu_daily,
    lw_cov_daily,
    objective="max_sharpe",
    min_weight=config.min_weight,
    max_weight=config.max_weight,
    seed=config.seed
)

weights = {
    "equal_weight": equal_w,
    "sample_gmv": sample_gmv_w,
    "lw_gmv": lw_gmv_w,
    "sample_max_sharpe": sample_sharpe_w,
    "lw_max_sharpe": lw_sharpe_w,
    "lw_mean_variance": lw_mv_w,
    "lw_shrunk_mu_max_sharpe": lw_shrunk_mu_sharpe_w,
}

weights_df = weight_table(weights)
weights_df

In [ ]:
plt.figure(figsize=(12, 7))
bottom = np.zeros(len(weights_df.columns))
x = np.arange(len(weights_df.columns))

for asset in weights_df.index:
    plt.bar(x, weights_df.loc[asset], bottom=bottom, label=asset)
    bottom += weights_df.loc[asset].values

plt.xticks(x, weights_df.columns, rotation=45, ha="right")
plt.title("Optimised Portfolio Weights")
plt.ylabel("Weight")
plt.legend(ncol=3, fontsize=8)
plt.tight_layout()
plt.show()

## 13. In-sample expected risk and return

Using each portfolio's own estimation inputs, compute:

$$
\hat\mu_p = w^\top\hat\mu
$$

$$
\hat\sigma_p = \sqrt{w^\top\hat\Sigma w}
$$

This is not final proof. It is only in-sample expectation.

In [ ]:
def portfolio_estimate_summary(weights, mu_daily, cov_daily, label):
    rows = []

    for name, w in weights.items():
        ann_return = portfolio_return(w, mu_daily) * config.annualisation
        ann_vol = portfolio_volatility(w, cov_daily) * np.sqrt(config.annualisation)
        sharpe = (ann_return - config.risk_free_rate_ann) / ann_vol if ann_vol > 0 else np.nan

        rows.append({
            "portfolio": name,
            "estimator": label,
            "expected_return_ann": ann_return,
            "expected_vol_ann": ann_vol,
            "expected_sharpe": sharpe,
            "gross_weight": float(np.sum(np.abs(w))),
            "max_weight": float(np.max(w)),
            "effective_n": float(1.0 / np.sum(w**2))
        })

    return pd.DataFrame(rows)


sample_est_summary = portfolio_estimate_summary(weights, sample_mu_daily, sample_cov_daily, "sample_inputs")
lw_est_summary = portfolio_estimate_summary(weights, sample_mu_daily, lw_cov_daily, "lw_cov_inputs")
true_summary = portfolio_estimate_summary(weights, true_mu_daily, true_cov_daily, "true_synthetic_inputs")

estimate_summary = pd.concat([sample_est_summary, lw_est_summary, true_summary], ignore_index=True)

estimate_summary.head()

In [ ]:
estimate_summary[estimate_summary["estimator"] == "true_synthetic_inputs"].sort_values("expected_sharpe", ascending=False)

## 14. Efficient frontier

We generate long-only target-return efficient frontier portfolios.

For each target return:

$$
\min_w \quad w^\top\Sigma w
$$

subject to:

$$
w^\top\mu=\mu^*
$$

$$
\sum_i w_i=1
$$

$$
0\le w_i\le w_{max}
$$

We compare sample covariance and Ledoit-Wolf covariance frontiers.

In [ ]:
def efficient_frontier(mu_daily, cov_daily, min_weight, max_weight, n_points=25):
    min_ret = float(np.percentile(mu_daily, 20))
    max_ret = float(np.percentile(mu_daily, 90))
    targets = np.linspace(min_ret, max_ret, n_points)

    rows = []

    for target in targets:
        w, info = optimise_long_only(
            mu_daily,
            cov_daily,
            objective="min_var",
            target_return=target,
            min_weight=min_weight,
            max_weight=max_weight,
            seed=config.seed
        )

        ann_ret = portfolio_return(w, mu_daily) * config.annualisation
        ann_vol = portfolio_volatility(w, cov_daily) * np.sqrt(config.annualisation)

        rows.append({
            "target_return_daily": target,
            "expected_return_ann": ann_ret,
            "expected_vol_ann": ann_vol,
            "success": info["success"],
            "method": info["method"],
            "weights": w
        })

    return pd.DataFrame(rows)


frontier_sample = efficient_frontier(sample_mu_daily, sample_cov_daily, config.min_weight, config.max_weight)
frontier_lw = efficient_frontier(sample_mu_daily, lw_cov_daily, config.min_weight, config.max_weight)

frontier_sample.head()

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(frontier_sample["expected_vol_ann"], frontier_sample["expected_return_ann"], marker="o", label="sample covariance")
plt.plot(frontier_lw["expected_vol_ann"], frontier_lw["expected_return_ann"], marker="x", label="Ledoit-Wolf covariance")

for name, w in weights.items():
    vol = portfolio_volatility(w, lw_cov_daily) * np.sqrt(config.annualisation)
    ret = portfolio_return(w, sample_mu_daily) * config.annualisation
    plt.scatter(vol, ret, s=70)
    plt.text(vol, ret, name, fontsize=8)

plt.title("Estimated Efficient Frontier")
plt.xlabel("Expected volatility")
plt.ylabel("Expected return")
plt.legend()
plt.show()

## 15. Out-of-sample performance

In-sample optimisation is cheap to impress.

The real test is out of sample.

We apply static train-estimated weights to the test period and compare:

- realised return;
- realised volatility;
- Sharpe proxy;
- max drawdown.

In [ ]:
def backtest_static_weights(test_returns, weights, transaction_cost_bps=0.0):
    rows = []
    bt = pd.DataFrame(index=test_returns.index)

    for name, w in weights.items():
        gross = test_returns @ w

        # Static one-time rebalance cost at first test day.
        cost = pd.Series(0.0, index=test_returns.index)
        cost.iloc[0] = np.sum(np.abs(w)) * transaction_cost_bps / 10000.0

        net = gross - cost
        nav = (1 + net).cumprod()

        bt[f"{name}_return"] = net
        bt[f"{name}_nav"] = nav

        dd = nav / nav.cummax() - 1

        rows.append({
            "portfolio": name,
            "realised_return_ann": float(net.mean() * config.annualisation),
            "realised_vol_ann": float(net.std(ddof=1) * np.sqrt(config.annualisation)),
            "realised_sharpe_proxy": float((net.mean() * config.annualisation - config.risk_free_rate_ann) / (net.std(ddof=1) * np.sqrt(config.annualisation))) if net.std(ddof=1) > 0 else np.nan,
            "max_drawdown": float(dd.min()),
            "total_return": float(nav.iloc[-1] - 1.0),
            "initial_cost": float(cost.iloc[0])
        })

    return bt, pd.DataFrame(rows).sort_values("realised_sharpe_proxy", ascending=False)


static_backtest, static_perf = backtest_static_weights(
    test_returns,
    weights,
    transaction_cost_bps=config.transaction_cost_bps
)

static_perf

In [ ]:
plt.figure(figsize=(12, 6))
for name in weights:
    plt.plot(static_backtest.index, static_backtest[f"{name}_nav"], label=name)
plt.title("Out-of-Sample Static Portfolio NAV")
plt.xlabel("Date")
plt.ylabel("Growth of 1")
plt.legend(ncol=2)
plt.show()

## 16. In-sample expectation versus realised test performance

We compare:

- expected Sharpe using train estimates;
- realised Sharpe in test.

This usually reveals optimisation optimism.

In [ ]:
expected_vs_realised = (
    estimate_summary[estimate_summary["estimator"] == "lw_cov_inputs"][["portfolio", "expected_return_ann", "expected_vol_ann", "expected_sharpe"]]
    .merge(static_perf[["portfolio", "realised_return_ann", "realised_vol_ann", "realised_sharpe_proxy"]], on="portfolio", how="left")
)

expected_vs_realised["sharpe_gap_realised_minus_expected"] = (
    expected_vs_realised["realised_sharpe_proxy"] - expected_vs_realised["expected_sharpe"]
)

expected_vs_realised.sort_values("sharpe_gap_realised_minus_expected")

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(expected_vs_realised["expected_sharpe"], expected_vs_realised["realised_sharpe_proxy"])

for _, row in expected_vs_realised.iterrows():
    plt.text(row["expected_sharpe"], row["realised_sharpe_proxy"], row["portfolio"], fontsize=8)

plt.axline((0, 0), slope=1, linestyle="--")
plt.title("Expected vs Realised Sharpe")
plt.xlabel("Expected Sharpe from train")
plt.ylabel("Realised Sharpe in test")
plt.show()

## 17. Sensitivity to expected returns

Mean-variance portfolios can change drastically when $\mu$ changes slightly.

We perturb expected returns and re-optimise max Sharpe.

If weights vary wildly, the optimisation is not robust.

In [ ]:
def mu_sensitivity_experiment(mu_daily, cov_daily, n_trials=100, noise_ann=0.03):
    rng = np.random.default_rng(config.seed + 500)
    rows = []
    weight_records = []

    noise_daily = noise_ann / config.annualisation

    for trial in range(n_trials):
        perturbed_mu = mu_daily + noise_daily * rng.standard_normal(len(mu_daily))

        w, info = optimise_long_only(
            perturbed_mu,
            cov_daily,
            objective="max_sharpe",
            min_weight=config.min_weight,
            max_weight=config.max_weight,
            seed=config.seed + trial
        )

        rows.append({
            "trial": trial,
            "method": info["method"],
            "expected_return_ann": portfolio_return(w, perturbed_mu) * config.annualisation,
            "expected_vol_ann": portfolio_volatility(w, cov_daily) * np.sqrt(config.annualisation),
            "effective_n": 1.0 / np.sum(w**2)
        })

        weight_records.append(pd.Series(w, index=asset_cols, name=trial))

    return pd.DataFrame(rows), pd.DataFrame(weight_records)


sens_sample, sens_weights_sample = mu_sensitivity_experiment(sample_mu_daily, sample_cov_daily)
sens_lw, sens_weights_lw = mu_sensitivity_experiment(sample_mu_daily, lw_cov_daily)

sensitivity_summary = pd.DataFrame({
    "asset": asset_cols,
    "sample_cov_weight_std": sens_weights_sample.std(),
    "lw_cov_weight_std": sens_weights_lw.std(),
    "sample_cov_mean_weight": sens_weights_sample.mean(),
    "lw_cov_mean_weight": sens_weights_lw.mean(),
})

sensitivity_summary.sort_values("lw_cov_weight_std", ascending=False)

In [ ]:
plt.figure(figsize=(10, 6))
plt.barh(sensitivity_summary["asset"], sensitivity_summary["sample_cov_weight_std"], alpha=0.6, label="sample cov")
plt.barh(sensitivity_summary["asset"], sensitivity_summary["lw_cov_weight_std"], alpha=0.6, label="Ledoit-Wolf")
plt.title("Weight Sensitivity to Expected Return Noise")
plt.xlabel("Weight standard deviation across perturbations")
plt.ylabel("Asset")
plt.legend()
plt.show()

## 18. Walk-forward optimisation

Static train/test is useful but incomplete.

We now run a walk-forward process:

1. estimate $\mu$ and $\Sigma$ using a rolling window;
2. optimise weights;
3. hold for one rebalance step;
4. repeat.

We compare sample covariance and Ledoit-Wolf covariance.

In [ ]:
def walk_forward_optimisation(returns, config, method="ledoit_wolf", objective="max_sharpe", shrink_mu=True):
    dates = returns.index
    rows = []
    weight_rows = []

    current = config.estimation_window

    while current < len(returns) - 1:
        train_window = returns.iloc[current - config.estimation_window:current]
        hold_start = current
        hold_end = min(current + config.rebalance_step, len(returns))

        mu = train_window.mean().to_numpy()

        if shrink_mu:
            mu = shrink_expected_returns(mu, shrinkage=0.50)

        if method == "ledoit_wolf":
            cov, shrinkage, cov_method = estimate_ledoit_wolf_covariance(train_window)
        elif method == "sample":
            cov = train_window.cov().to_numpy()
            shrinkage = 0.0
            cov_method = "sample"
        else:
            raise ValueError("method must be sample or ledoit_wolf.")

        w, info = optimise_long_only(
            mu,
            cov,
            objective=objective,
            min_weight=config.min_weight,
            max_weight=config.max_weight,
            risk_aversion=config.risk_aversion,
            seed=config.seed + current
        )

        for date in dates[hold_start:hold_end]:
            weight_rows.append(pd.Series(w, index=asset_cols, name=date))

        rows.append({
            "rebalance_date": dates[current],
            "method": method,
            "objective": objective,
            "shrinkage": shrinkage,
            "optimizer_method": info["method"],
            "effective_n": float(1.0 / np.sum(w**2)),
            "max_weight": float(np.max(w)),
            "gross_weight": float(np.sum(np.abs(w)))
        })

        current += config.rebalance_step

    weights_wf = pd.DataFrame(weight_rows)
    weights_wf.index.name = "date"
    weights_wf = weights_wf.reindex(returns.index).ffill().fillna(0.0)

    diagnostics = pd.DataFrame(rows)

    return weights_wf, diagnostics


wf_weights_sample, wf_diag_sample = walk_forward_optimisation(returns, config, method="sample", objective="max_sharpe", shrink_mu=True)
wf_weights_lw, wf_diag_lw = walk_forward_optimisation(returns, config, method="ledoit_wolf", objective="max_sharpe", shrink_mu=True)

wf_diag_lw.head()

## 19. Walk-forward backtest with turnover costs

Weights decided at rebalance date are applied to next-day returns.

Costs are proportional to weight changes:

$$
cost_t = c\sum_i |w_{i,t}-w_{i,t-1}|
$$

In [ ]:
def backtest_dynamic_weights(returns, weights, cost_bps, initial_capital=1.0):
    investable = weights.shift(1).fillna(0.0)
    gross = (investable * returns).sum(axis=1)

    turnover = weights.diff().abs().sum(axis=1).fillna(weights.abs().sum(axis=1))
    cost = turnover * cost_bps / 10000.0

    net = gross - cost
    nav = initial_capital * (1 + net).cumprod()

    return pd.DataFrame({
        "gross_return": gross,
        "transaction_cost": cost,
        "net_return": net,
        "nav": nav,
        "turnover": turnover,
        "gross_exposure": investable.abs().sum(axis=1)
    })


wf_bt_sample = backtest_dynamic_weights(returns, wf_weights_sample, config.transaction_cost_bps)
wf_bt_lw = backtest_dynamic_weights(returns, wf_weights_lw, config.transaction_cost_bps)
wf_bt_equal = backtest_dynamic_weights(returns, pd.DataFrame(equal_w, index=returns.index, columns=asset_cols), config.transaction_cost_bps)

walkforward_backtests = {
    "wf_sample_cov": wf_bt_sample,
    "wf_ledoit_wolf": wf_bt_lw,
    "equal_weight": wf_bt_equal,
}

def summarise_backtest_dict(backtests):
    rows = []

    for name, bt in backtests.items():
        r = bt["net_return"]
        nav = bt["nav"]
        dd = nav / nav.cummax() - 1

        rows.append({
            "strategy": name,
            "annualised_return": float(r.mean() * config.annualisation),
            "annualised_vol": float(r.std(ddof=1) * np.sqrt(config.annualisation)),
            "sharpe_proxy": float((r.mean() * config.annualisation - config.risk_free_rate_ann) / (r.std(ddof=1) * np.sqrt(config.annualisation))) if r.std(ddof=1) > 0 else np.nan,
            "max_drawdown": float(dd.min()),
            "total_return": float(nav.iloc[-1] - 1.0),
            "mean_turnover": float(bt["turnover"].mean()),
            "annualised_cost_drag": float(bt["transaction_cost"].mean() * config.annualisation),
            "mean_gross_exposure": float(bt["gross_exposure"].mean()),
        })

    return pd.DataFrame(rows).sort_values("sharpe_proxy", ascending=False)


wf_perf = summarise_backtest_dict(walkforward_backtests)
wf_perf

In [ ]:
plt.figure(figsize=(12, 6))
for name, bt in walkforward_backtests.items():
    plt.plot(bt.index, bt["nav"], label=name)
plt.title("Walk-Forward Optimisation Backtest")
plt.xlabel("Date")
plt.ylabel("Growth of 1")
plt.legend()
plt.show()

plt.figure(figsize=(12, 5))
plt.plot(wf_bt_sample.index, wf_bt_sample["turnover"].rolling(21).mean(), label="sample covariance")
plt.plot(wf_bt_lw.index, wf_bt_lw["turnover"].rolling(21).mean(), label="Ledoit-Wolf")
plt.title("Rolling Average Turnover")
plt.xlabel("Date")
plt.ylabel("Turnover")
plt.legend()
plt.show()

## 20. Walk-forward weights

We inspect whether Ledoit-Wolf shrinkage produces more stable allocations.

In [ ]:
weight_stability = pd.DataFrame({
    "asset": asset_cols,
    "sample_mean_weight": wf_weights_sample.mean(),
    "sample_weight_std": wf_weights_sample.std(),
    "lw_mean_weight": wf_weights_lw.mean(),
    "lw_weight_std": wf_weights_lw.std(),
}).reset_index(drop=True)

weight_stability.sort_values("lw_weight_std", ascending=False)

In [ ]:
plt.figure(figsize=(12, 6))
for asset in asset_cols:
    plt.plot(wf_weights_lw.index, wf_weights_lw[asset], label=asset, alpha=0.75)
plt.title("Walk-Forward Ledoit-Wolf Weights")
plt.xlabel("Date")
plt.ylabel("Weight")
plt.legend(ncol=3)
plt.show()

## 21. Realised risk attribution

For a portfolio with weights $w_t$, a simple variance contribution approximation is:

$$
RC_i = \frac{ w_i(\Sigma w)_i }{ w^\top\Sigma w }
$$

We compute average risk contributions for the latest Ledoit-Wolf portfolio.

In [ ]:
def risk_contributions(w, cov):
    w = np.asarray(w, dtype=float)
    marginal = cov @ w
    total_var = w.T @ cov @ w

    if total_var <= 0:
        return np.zeros_like(w)

    return w * marginal / total_var


latest_w_lw = wf_weights_lw.iloc[-1].to_numpy()
latest_window = returns.iloc[-config.estimation_window:]
latest_cov, latest_shrinkage, _ = estimate_ledoit_wolf_covariance(latest_window)

latest_rc = risk_contributions(latest_w_lw, latest_cov)

risk_contribution_table = pd.DataFrame({
    "asset": asset_cols,
    "latest_weight": latest_w_lw,
    "latest_risk_contribution": latest_rc
}).sort_values("latest_risk_contribution", ascending=False)

risk_contribution_table

In [ ]:
plt.figure(figsize=(10, 6))
plt.barh(risk_contribution_table.sort_values("latest_risk_contribution")["asset"], risk_contribution_table.sort_values("latest_risk_contribution")["latest_risk_contribution"])
plt.title("Latest Ledoit-Wolf Portfolio Risk Contributions")
plt.xlabel("Risk contribution")
plt.ylabel("Asset")
plt.show()

## 22. Constraint diagnostics

Mean-variance optimisers often sit on constraints.

We check:

- maximum weight usage;
- number of assets near max weight;
- effective number of bets;
- concentration.

In [ ]:
def constraint_diagnostics(weights_df, max_weight):
    rows = []

    for date, row in weights_df.iterrows():
        w = row.to_numpy()

        rows.append({
            "date": date,
            "max_weight": float(np.max(w)),
            "n_at_max_weight": int(np.sum(w >= max_weight - 1e-6)),
            "n_active": int(np.sum(w > 1e-6)),
            "effective_n": float(1.0 / np.sum(w**2)) if np.sum(w**2) > 0 else np.nan,
            "herfindahl": float(np.sum(w**2))
        })

    return pd.DataFrame(rows)


constraint_diag_lw = constraint_diagnostics(wf_weights_lw, config.max_weight)
constraint_diag_sample = constraint_diagnostics(wf_weights_sample, config.max_weight)

constraint_summary = pd.DataFrame([
    {
        "strategy": "sample_cov",
        "mean_effective_n": constraint_diag_sample["effective_n"].mean(),
        "mean_n_at_max": constraint_diag_sample["n_at_max_weight"].mean(),
        "mean_max_weight": constraint_diag_sample["max_weight"].mean()
    },
    {
        "strategy": "ledoit_wolf",
        "mean_effective_n": constraint_diag_lw["effective_n"].mean(),
        "mean_n_at_max": constraint_diag_lw["n_at_max_weight"].mean(),
        "mean_max_weight": constraint_diag_lw["max_weight"].mean()
    }
])

constraint_summary

## 23. Practical checklist for MVO

Before trusting an optimised portfolio, check:

1. **Input stability**  
   Are $\mu$ and $\Sigma$ stable?

2. **Covariance conditioning**  
   Is the covariance matrix ill-conditioned?

3. **Shrinkage**  
   Does shrinkage reduce instability?

4. **Expected return sensitivity**  
   Do small changes in $\mu$ cause huge weight changes?

5. **Constraints**  
   Are weights sitting on bounds?

6. **Out-of-sample performance**  
   Does the optimiser beat equal weight after costs?

7. **Turnover**  
   Are rebalances too aggressive?

8. **Risk contributions**  
   Is the portfolio secretly dominated by one asset?

9. **Stress regimes**  
   Does the optimiser concentrate before stress?

10. **Economic rationale**  
   Are allocations explainable, or just numerical artefacts?

## 24. Saving outputs

The notebook saves:

1. synthetic returns;
2. asset metadata;
3. expected-return estimates;
4. covariance diagnostics;
5. Ledoit-Wolf covariance;
6. optimised weights;
7. efficient frontier;
8. static out-of-sample backtest;
9. sensitivity experiments;
10. walk-forward weights;
11. walk-forward backtests;
12. risk contribution report;
13. constraint diagnostics;
14. manifest.

In [ ]:
output_dir = Path("data/processed/mean_variance_optimization_ledoit_wolf_v1")
output_dir.mkdir(parents=True, exist_ok=True)

returns_path = output_dir / "synthetic_returns.csv"
metadata_path = output_dir / "asset_metadata.csv"
mu_estimates_path = output_dir / "mu_estimates.csv"
mu_shrink_path = output_dir / "mu_shrink_table.csv"
sample_cov_path = output_dir / "sample_covariance_annualized.csv"
lw_cov_path = output_dir / "ledoit_wolf_covariance_annualized.csv"
true_cov_path = output_dir / "true_covariance_annualized.csv"
cov_diag_path = output_dir / "covariance_diagnostics.csv"
weights_path = output_dir / "optimised_weights.csv"
estimate_summary_path = output_dir / "estimate_summary.csv"
frontier_sample_path = output_dir / "frontier_sample.csv"
frontier_lw_path = output_dir / "frontier_ledoit_wolf.csv"
static_perf_path = output_dir / "static_out_of_sample_performance.csv"
static_backtest_path = output_dir / "static_backtest.csv"
expected_vs_realised_path = output_dir / "expected_vs_realised.csv"
sensitivity_summary_path = output_dir / "mu_sensitivity_summary.csv"
wf_weights_sample_path = output_dir / "walkforward_weights_sample_cov.csv"
wf_weights_lw_path = output_dir / "walkforward_weights_ledoit_wolf.csv"
wf_diag_sample_path = output_dir / "walkforward_diag_sample.csv"
wf_diag_lw_path = output_dir / "walkforward_diag_ledoit_wolf.csv"
wf_perf_path = output_dir / "walkforward_performance.csv"
weight_stability_path = output_dir / "weight_stability.csv"
risk_contrib_path = output_dir / "latest_risk_contribution.csv"
constraint_summary_path = output_dir / "constraint_summary.csv"
constraint_diag_lw_path = output_dir / "constraint_diag_ledoit_wolf.csv"
manifest_path = output_dir / "manifest.json"

returns_df.to_csv(returns_path, index=False)
asset_metadata.to_csv(metadata_path, index=False)
mu_estimates.to_csv(mu_estimates_path, index=False)
mu_shrink_table.to_csv(mu_shrink_path, index=False)
pd.DataFrame(sample_cov_ann, index=asset_cols, columns=asset_cols).to_csv(sample_cov_path)
pd.DataFrame(lw_cov_ann, index=asset_cols, columns=asset_cols).to_csv(lw_cov_path)
pd.DataFrame(true_cov_ann, index=asset_cols, columns=asset_cols).to_csv(true_cov_path)
cov_diag_table.to_csv(cov_diag_path, index=False)
weights_df.to_csv(weights_path)
estimate_summary.to_csv(estimate_summary_path, index=False)
frontier_sample.drop(columns=["weights"]).to_csv(frontier_sample_path, index=False)
frontier_lw.drop(columns=["weights"]).to_csv(frontier_lw_path, index=False)
static_perf.to_csv(static_perf_path, index=False)
static_backtest.to_csv(static_backtest_path)
expected_vs_realised.to_csv(expected_vs_realised_path, index=False)
sensitivity_summary.to_csv(sensitivity_summary_path, index=False)
wf_weights_sample.to_csv(wf_weights_sample_path)
wf_weights_lw.to_csv(wf_weights_lw_path)
wf_diag_sample.to_csv(wf_diag_sample_path, index=False)
wf_diag_lw.to_csv(wf_diag_lw_path, index=False)
wf_perf.to_csv(wf_perf_path, index=False)
weight_stability.to_csv(weight_stability_path, index=False)
risk_contribution_table.to_csv(risk_contrib_path, index=False)
constraint_summary.to_csv(constraint_summary_path, index=False)
constraint_diag_lw.to_csv(constraint_diag_lw_path, index=False)

manifest = {
    "dataset_name": "mean_variance_optimization_ledoit_wolf_outputs",
    "schema_version": "mean_variance_optimization_ledoit_wolf_v1",
    "created_at": pd.Timestamp.now(tz="UTC").isoformat(),
    "config": asdict(config),
    "asset_cols": asset_cols,
    "sklearn_available": SKLEARN_AVAILABLE,
    "scipy_available": SCIPY_AVAILABLE,
    "ledoit_wolf_method": lw_method,
    "ledoit_wolf_shrinkage": lw_shrinkage,
    "covariance_diagnostics": cov_diag_table.to_dict(orient="records"),
    "static_out_of_sample_performance": static_perf.to_dict(orient="records"),
    "walkforward_performance": wf_perf.to_dict(orient="records"),
    "notes": [
        "Ledoit-Wolf shrinkage is used via sklearn when available; otherwise a fixed identity-shrinkage fallback is used.",
        "Expected returns are shown as noisy and optionally shrunk toward the grand mean.",
        "Optimisation is long-only with max-weight constraints.",
        "Efficient frontiers are estimated in sample; final judgement is out of sample.",
        "Walk-forward optimisation includes turnover costs.",
        "This notebook emphasises robustness and diagnostics rather than frontier aesthetics."
    ]
}

manifest_path.write_text(json.dumps(manifest, indent=2, default=str))

returns_path, weights_path, static_perf_path, wf_perf_path, manifest_path

## 25. Limitations

### 25.1 Synthetic data

The notebook uses synthetic returns.

Real portfolios need clean total-return data, corporate actions, futures rolls, FX conversion, and investability filters.

### 25.2 Expected returns are unreliable

MVO is extremely sensitive to $\mu$.

This notebook demonstrates shrinkage but does not solve expected-return forecasting.

### 25.3 Covariance shrinkage is not a cure-all

Ledoit-Wolf stabilises covariance but cannot fix bad expected returns, structural breaks, or missing risk factors.

### 25.4 Long-only constraints

The optimiser is long-only.

Long-short portfolios require leverage, borrow, margin, and financing constraints.

### 25.5 Simple transaction costs

Costs are fixed bps.

Real costs depend on spread, liquidity, participation, market impact, and volatility.

### 25.6 Efficient frontier is in-sample

The frontier can look good even when out-of-sample performance is poor.

### 25.7 No robust optimisation

This notebook does not implement uncertainty sets or Bayesian posterior predictive optimisation.

## 26. Research frontier and extensions

Important extensions include:

1. **Black-Litterman optimisation**  
   Blend equilibrium returns with investor views.

2. **Robust mean-variance optimisation**  
   Optimise under parameter uncertainty.

3. **Bayesian covariance estimation**  
   Use posterior covariance distributions.

4. **Hierarchical risk parity**  
   Avoid direct inversion of covariance.

5. **Expected-return shrinkage**  
   Use factor models, IC estimates, or Bayesian priors.

6. **Turnover-constrained optimisation**  
   Penalise deviation from current weights.

7. **Transaction-cost-aware optimisation**  
   Optimise net expected return after costs.

8. **Full futures constraints**  
   Include margin, contract multipliers, tick values, and roll rules.

9. **Stress-aware covariance**  
   Blend normal and stress covariance matrices.

10. **Chinese futures portfolio optimisation**  
   Apply shrinkage covariance to multi-commodity futures with roll, liquidity, and night-session effects.

## 27. Suggested follow-up notebooks

This notebook naturally leads to:

1. **04_05_risk_parity_and_equal_risk_contribution**  
   Avoid expected-return sensitivity by focusing on risk allocation.

2. **04_06_black_litterman_allocation**  
   Stabilise expected returns with priors and views.

3. **04_07_var_cvar_and_stress_testing**  
   Tail-risk diagnostics for optimised portfolios.

4. **04_08_portfolio_constraints_and_turnover_control**  
   Make optimisation realistic.

5. **05_01_vectorized_backtest_engine**  
   Use this optimiser in a reusable backtest engine.

6. **07_01_multi_asset_cta_research_pipeline**  
   Use shrinkage covariance in a full futures allocation system.

## 28. Summary

This notebook implemented mean-variance optimisation with Ledoit-Wolf shrinkage.

It showed how to:

1. simulate multi-asset returns with true covariance;
2. estimate sample expected returns and covariance;
3. diagnose covariance instability;
4. estimate Ledoit-Wolf shrinkage covariance;
5. shrink expected returns;
6. optimise global minimum variance and maximum Sharpe portfolios;
7. apply long-only and max-weight constraints;
8. build efficient frontiers;
9. compare estimated versus realised out-of-sample performance;
10. test sensitivity to expected-return noise;
11. run walk-forward optimisation;
12. include turnover costs;
13. compute risk contributions;
14. diagnose constraint behaviour;
15. save outputs and metadata.

The key computational takeaway:

> Stable covariance estimation is a prerequisite for useful mean-variance optimisation.

The key financial takeaway:

> The optimiser is only as good as its inputs; shrinkage reduces one source of noise, but it does not turn noisy expected returns into truth.

## 29. Further reading

- Markowitz, *Portfolio Selection.*
- Ledoit and Wolf on covariance shrinkage.
- Jorion on Bayes-Stein shrinkage.
- Black and Litterman on global portfolio optimisation.
- Michaud on estimation error and resampled efficient frontiers.
- Meucci, *Risk and Asset Allocation.*